In [ ]:
from langchain_core.documents import Document
import json
from google import genai
from langchain_qdrant import QdrantVectorStore,FastEmbedSparse,RetrievalMode
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings
from langsmith import wrappers
from google.genai import types
from langsmith import traceable
from dotenv import load_dotenv
from typing import Optional
from langchain_huggingface import HuggingFaceEmbeddings
import random
import cohere
import time
import random
from google import genai
from pydantic import BaseModel, Field
from pathlib import Path
load_dotenv()



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
try:
    base_path = Path(__file__).parent
    with open(base_path.parent / 'json_files' /'final_data.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
except FileNotFoundError:
    print("Error: The file final_data.json does not exist.")  
    


Error: The file final_data.json does not exist.


In [ ]:
# ###creating seperate dict for chunks with images , adding img summary,url along with metadata to the dict
# img_chunks=[]
# for ele in data:
#     if ele['img_summary']:
#         img_chunks.append({
#             'eid':f"{ele['eid']}_img",
#             'metadata':ele['metadata'],
#             'text':ele.pop('img_summary'),
#             'img':ele.pop('img')
#         })

In [ ]:
##removing img_summary, table, img_url after moving images and text to other dict
# data=[{key: val for key, val in ele.items() if val} for ele in data ]

In [14]:
# final_list=data+img_chunks
# len(final_list)
len(data)

471

In [18]:
##document creation
docu_list=[]
for element in data:
    extra_data={'img_url':element['img_url'],'eid':element['id']}
    doc=Document(page_content=element['text'],
             metadata=element['metadata']|extra_data
             )
    docu_list.append(doc)
docs_list=[{
    "page_content":docs.page_content,
    "metadata":docs.metadata

}for docs in docu_list]    
with open("langchain_docu.json", "w") as f:
    json.dump(docs_list, f, indent=2)


In [9]:
dense_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5", model_kwargs={'device':'mps'})
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/minicoil-v1",kwargs={'device': 'mps'})




Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6276.25it/s]


In [4]:
co=cohere.ClientV2()

In [5]:
client = QdrantClient(url="http://localhost:6333")
collection_name="RAG_PROJECT"

In [10]:
if client.collection_exists(collection_name):
    print("Collection found! Loading existing vectors...")
    vector_store = QdrantVectorStore.from_existing_collection(
        embedding=dense_embeddings,
        sparse_embedding=sparse_embeddings,
        url="http://localhost:6333",
        collection_name=collection_name,
        retrieval_mode=RetrievalMode.HYBRID
        
    )
else:
    print("Collection not found. Initializing and embedding documents...")
    # This runs ONLY THE FIRST TIME to create and populate the DB
    vector_store = QdrantVectorStore.from_documents(
        documents=docu_list,
        embedding=dense_embeddings,
        sparse_embedding=sparse_embeddings,
        url="http://localhost:6333",
        collection_name=collection_name,
        retrieval_mode=RetrievalMode.HYBRID
    )


 

Collection found! Loading existing vectors...


In [11]:
temp=vector_store.similarity_search_with_score("what is cot",k=5 )
print(vector_store)

In [31]:

@traceable(name="Chunks Retrieved", metadata={"search_type": "hybrid"})
def retrieval(user_query,k_val):
    results = vector_store.similarity_search_with_score(user_query,k=k_val)
    results=[result for result in results if result[1]>0.2]
    return(results)
 


    


In [32]:
@traceable
def rerank(chunks,user_query,final_k):
    text_chunks=[chunk for chunk in chunks if "_txt" in chunk[0].metadata['eid'] ]
    img_chunks=[chunk for chunk in chunks if "_txt" not in chunk[0].metadata['eid']]
    text_passage=[chunk[0].page_content for chunk in text_chunks]
    response = co.rerank(
    model="rerank-english-v3.0",
    query=user_query,
    documents=text_passage,
    top_n=final_k,
)
    top_text = [text_chunks[r.index] for r in response.results]
    return top_text + img_chunks[:1]
    

    

In [33]:
@traceable
def genereate(results,user_query):
    context="\n\n\n".join([f"Page Content:{result[0].page_content}\n Page Number:{result[0].metadata['pages']}\n File Location:{result[0].metadata['filename']}"
                        for result in results])
    SYSTEM_PROMPT=f"""
    You are a helpful AI assistant who answers user query based on available context, retrieved from pdf file.
    You should ONLY ans the user based on the following context and help user navigate to the right page number and to the right file to know more, ALWAYS mention what detail is available on which page. IF u find no chunk matching the query ONLY send that u didn't find anything 
    Context
    {context}
    """
    client = genai.Client()
    wrapped_client = wrappers.wrap_gemini(
            client,
            tracing_extra={
                "tags": ["gemini", "python"],
                "metadata": {
                    "integration": "google-genai",
                },
            },
    )    
    interaction = wrapped_client.models.generate_content(
        model="gemini-3.1-flash-lite", 
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
        ),
        contents=user_query
    )

    return(interaction.text)


In [36]:
final_k=3

In [ ]:
# @traceable(name="with re_ranker")
# def pipepline(user_query,k_val):
#     chunks=retrieval(user_query,k_val)
#     rerank_chunks=rerank(chunks,user_query,final_k)
#     ans=genereate(rerank_chunks,user_query)
#     return(ans)
    
       
    

In [34]:
@traceable(name="with re_ranker")
def pipepline(user_query,k_val):
    chunks=retrieval(user_query,k_val)
    rerank_chunks=rerank(chunks,user_query,final_k)
    ans=genereate(rerank_chunks,user_query)
    return(ans)

In [ ]:
# def pipepline(user_query,k_val):
#     chunks=retrieval(user_query,k_val)
#     rerank_chunks=rerank(chunks,user_query,final_k)

In [35]:
user_query="explain figure 11.12 from RAG.pdf"
k_val=15
pipepline(user_query,k_val)

NameError: name 'final_k' is not defined

In [192]:
docu_list[0].metadata['eid']

'3cef728d-d565-4b44-8bdb-977c9f603df7_txt'

In [ ]:
text_docu_list=[element for element in docu_list if '_txt' in element.metadata['eid'] ]


client = genai.Client()  


class ChunkAssessment(BaseModel):
    has_enough_context: bool
    reason: str
    question: Optional[str] = None
    ground_truth: Optional[str] = None

def generate_qa(chunk, max_retries=3):
    prompt = f"""Assess if this text chunk has enough self-contained context 
to generate a meaningful factual question without needing surrounding context.

If yes → generate a question and answer ONLY based on information 
present in this chunk. The question must be answerable from this chunk alone,
without needing any other text or context.
Do NOT generate questions that require comparing or combining 
information from multiple sources.
If no → set question and ground_truth to null.

Text: {chunk.page_content}"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": ChunkAssessment,
                }
            )
            
            result = response.parsed  # directly gives ChunkAssessment object
            
            if not result.has_enough_context:
                print(f"⏭️ Skipping: {result.reason}")
                return None
            
            return {
                "question": result.question,
                "ground_truth": result.ground_truth,
                "source": chunk.metadata.get("filename"),
                "page": chunk.metadata.get("pages")
            }
            
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # exponential backoff: 1s, 2s, 4s
                print(f"⚠️ Attempt {attempt+1} failed: {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"❌ Failed after {max_retries} attempts: {e}")
                return None

# main loop
dataset = []
target_questions=25
random.shuffle(text_docu_list)

for i, chunk in enumerate(text_docu_list):
    if len(dataset) >= target_questions:
        print(f"✅ Reached target of {target_questions} questions")
        break
    print(f"Processing {i+1}/{len(text_docu_list)}...")
    
    qa = generate_qa(chunk)
    if qa:
        dataset.append(qa)
        print(f"✅ {qa['question'][:60]}")
    
   
    time.sleep(4)

print(f"\nGenerated {len(dataset)} questions")



Processing 1/390...
✅ In the vector space model of information retrieval, how is a
Processing 2/390...
✅ In causal self-attention, what range of inputs does the mode
Processing 3/390...
✅ What kind of private information can an adversary potentiall
Processing 4/390...
✅ What will Chapter 12 of the book introduce?
Processing 5/390...
✅ Which 1818 novel is mentioned as an example of ethical and s
Processing 6/390...
✅ Why do words that are semantically close have no relation in
Processing 7/390...
✅ What is the token_id associated with the token 'HTML'?
Processing 8/390...
✅ What is the primary goal of transformers at each layer?
Processing 9/390...
✅ Why do language models generate incorrect or untrue text?
Processing 10/390...
✅ According to Figure 11.6, what is the tf-idf cosine score fo
Processing 11/390...
✅ What is the trade-off associated with methods that give more
Processing 12/390...
✅ What is the primary reason that the use of stop lists is les
Processing 13/390...
✅ Why do mo

In [43]:
# with open("ragas_dataset.json", "w") as f:
#     json.dump(dataset, f, indent=2)
dataset

[{'question': 'In the vector space model of information retrieval, how is a document represented?',
  'ground_truth': 'A document is represented as a vector of counts of the words it contains.',
  'source': 'RAG.pdf',
  'page': [4]},
 {'question': 'In causal self-attention, what range of inputs does the model have access to when processing a specific item?',
  'ground_truth': 'The model has access to all inputs up to and including the one under consideration, but no access to information about inputs beyond the current one.',
  'source': 'Transformers.pdf',
  'page': [4]},
 {'question': 'What kind of private information can an adversary potentially extract from a large language model?',
  'ground_truth': 'An adversary can extract training-data text from a language model, such as a person’s name, phone number, and address.',
  'source': 'LLM.pdf',
  'page': [23, 24]},
 {'question': 'What will Chapter 12 of the book introduce?',
  'ground_truth': 'Chapter 12 will introduce prompting larg

In [48]:
@traceable(name="RAG Evaluation Pipeline")
def eval_pipeline(item):
    chunks = retrieval(item["question"], k_val=20)
    reranked = rerank(chunks, item["question"], final_k=5)
    answer = genereate(reranked, item["question"])
    
    return {
        "question": item["question"],
        "answer": answer,
        "contexts": [c[0].page_content for c in reranked],
        "ground_truth": item["ground_truth"],
        "source":item['source'],
        "page_number":item['page']
    }

eval_dataset = []

for item in dataset:
    print(f"Running: {item['question'][:60]}")
    result = eval_pipeline(item)
    eval_dataset.append(result)
    time.sleep(4)

Running: In the vector space model of information retrieval, how is a
Running: In causal self-attention, what range of inputs does the mode
Running: What kind of private information can an adversary potentiall
Running: What will Chapter 12 of the book introduce?
Running: Which 1818 novel is mentioned as an example of ethical and s
Running: Why do words that are semantically close have no relation in
Running: What is the token_id associated with the token 'HTML'?
Running: What is the primary goal of transformers at each layer?
Running: Why do language models generate incorrect or untrue text?
Running: According to Figure 11.6, what is the tf-idf cosine score fo
Running: What is the trade-off associated with methods that give more
Running: What is the primary reason that the use of stop lists is les
Running: Why do most people prefer sampling methods over greedy decod
Running: Why can a short query and a long document be considered sema
Running: How is the logit modified in temperature s

In [49]:
eval_dataset

[{'question': 'In the vector space model of information retrieval, how is a document represented?',
  'answer': 'In the vector space model of information retrieval, a document is represented as a vector of counts of the words it contains. This is often referred to as a "bag-of-words" model, where the document is treated as an unordered set of words, ignoring their original position and focusing solely on their frequency.\n\nFor more details, you can refer to **page 4** of **RAG.pdf**.',
  'contexts': ['11.1.1 Representing documents as vectors\n\nvector space model\n\nmodel In the vector space model of information retrieval (Salton, 1971) a document is represented as a vector of counts of the words it contains.\n\nbag of words\n\nWe sometimes call this kind of model a bag-of-words model. Fig. 11.2 shows the intuition: we are representing a text document as if it were a bag of words, that is, an unordered set of words with their position ignored, keeping only their frequency in the docum

In [50]:
with open("eval_dataset.json", "w") as f:
    json.dump(eval_dataset, f, indent=2)